All of the packages used extensively in the pipeline

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import tensorflow as tf
import pickle
from scipy.special import boxcox

from sklearn.model_selection import train_test_split, cross_val_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from keras import callbacks

import sys
import unittest
import import_ipynb
import malwarefunctions

2024-06-28 14:06:14.820218: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-06-28 14:06:14.877302: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-06-28 14:06:14.878724: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-06-28 14:06:15.839306: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Tests preprocessing steps of dataframe such as normalization of numeric columns, dropping unnecessary columns, and dataframe's dimensions

In [2]:
def test_preprocess():
    class TestRunModel(unittest.TestCase):
        def test_df_shape(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            self.assertEqual(df.shape, (138047, 57))
        def test_value_counts(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            self.assertEqual(df['legitimate'].value_counts()[0], 96724)
            self.assertEqual(df['legitimate'].value_counts()[1], 41323)
        def test_normalization(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X = df.drop(columns=['legitimate'])
            for label, content in X.describe().items():
                if label != "legitimate":
                    self.assertAlmostEqual(content['mean'], 0)
                    self.assertAlmostEqual(content['std'], 1)
        def test_components_drop(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            self.assertEqual(df.shape, (138047, 37))
        
    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

importing Jupyter notebook from malwarefunctions.ipynb
Logistic Regression classifier:
Time taken to train the Logistic Regression model: 1.2695109844207764 seconds
Time taken to test the Logistic Regression model: 0.005425453186035156 seconds
Accuracy: 0.9784860557768924
Precision: 0.9696413269678051
Recall: 0.9581468489173823
F1 score: 0.9638598199075201


Tests training, test, and accuracy of Gradient Boosting model after preprocessing

In [3]:
def test_gbc():
    class TestRunModel(unittest.TestCase):
        def test_gbc_training(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = GradientBoostingClassifier(random_state=42)
            model.fit(X_train, y_train)
            self.assertIsNotNone(model)
            self.assertTrue(hasattr(model, 'feature_importances_'))
        def test_gbc_prediction(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = GradientBoostingClassifier(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            self.assertEqual(len(predictions), len(X_test))
        def test_gbc_accuracy(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = GradientBoostingClassifier(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)
            f1 = f1_score(y_test, predictions)
            self.assertGreaterEqual(accuracy, 0.9)
            self.assertGreaterEqual(f1, 0.9)

    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

Tests training, test, and accuracy of Gaussian Naive Bayes model after preprocessing

In [4]:
def test_nb():
    class TestRunModel(unittest.TestCase):
        def test_nb_training(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = GaussianNB()
            model.fit(X_train, y_train)
            self.assertIsNotNone(model)
            self.assertTrue(hasattr(model, 'class_prior_'))
        def test_nb_prediction(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = GaussianNB()
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            self.assertEqual(len(predictions), len(X_test))
        def test_nb_accuracy(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = GaussianNB()
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)
            f1 = f1_score(y_test, predictions)
            self.assertGreaterEqual(accuracy, 0.9)
            self.assertGreaterEqual(f1, 0.9)

    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

Tests training, test, and accuracy of Artificial Neural Network model after preprocessing

In [5]:
def test_ann():
    class TestRunModel(unittest.TestCase):
        def test_ann_training(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            input_shape = [X_train.shape[1]]
            model = tf.keras.Sequential([
                tf.keras.layers.Dense(units=64, activation='relu', input_shape=input_shape),
                tf.keras.layers.Dense(units=64, activation='relu'),
                tf.keras.layers.Dense(units=1)
            ])
            model.build()
            model.compile(optimizer='adam', loss='mae', metrics=['accuracy'])  
            earlystopping = callbacks.EarlyStopping(monitor="val_loss",
                                                        mode="min",
                                                        patience=5,
                                                        restore_best_weights=True)
            history = model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=256, epochs=60,callbacks=[earlystopping], verbose=0)
            self.assertIsNotNone(model)
        def test_ann_prediction(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            input_shape = [X_train.shape[1]]
            model = tf.keras.Sequential([
                tf.keras.layers.Dense(units=64, activation='relu', input_shape=input_shape),
                tf.keras.layers.Dense(units=64, activation='relu'),
                tf.keras.layers.Dense(units=1)
            ])
            model.build()
            model.compile(optimizer='adam', loss='mae', metrics=['accuracy'])  
            earlystopping = callbacks.EarlyStopping(monitor="val_loss",
                                                        mode="min",
                                                        patience=5,
                                                        restore_best_weights=True)
            history = model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=256, epochs=60,callbacks=[earlystopping], verbose=0)
            predictions = (model.predict(X_test) > 0.5).astype("int32")
            self.assertEqual(len(predictions), len(X_test))
        def test_ann_accuracy(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            input_shape = [X_train.shape[1]]
            model = tf.keras.Sequential([
                tf.keras.layers.Dense(units=64, activation='relu', input_shape=input_shape),
                tf.keras.layers.Dense(units=64, activation='relu'),
                tf.keras.layers.Dense(units=1)
            ])
            model.build()
            model.compile(optimizer='adam', loss='mae', metrics=['accuracy'])  
            earlystopping = callbacks.EarlyStopping(monitor="val_loss",
                                                        mode="min",
                                                        patience=5,
                                                        restore_best_weights=True)
            history = model.fit(X_train, y_train, validation_data=(X_test,y_test), batch_size=256, epochs=60,callbacks=[earlystopping], verbose=0)
            self.assertGreater(history.history['accuracy'][-1], 0.9)

    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

Tests training, test, and accuracy of Support Vector Machine model after preprocessing

In [6]:
def test_svm():
    class TestRunModel(unittest.TestCase):
        def test_svm_training(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = SVC(random_state=42)
            model.fit(X_train, y_train)
            self.assertIsNotNone(model)
            self.assertTrue(hasattr(model, 'class_weight_'))
        def test_svm_prediction(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = SVC(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            self.assertEqual(len(predictions), len(X_test))
        def test_svm_accuracy(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = SVC(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)
            f1 = f1_score(y_test, predictions)
            self.assertGreaterEqual(accuracy, 0.9)
            self.assertGreaterEqual(f1, 0.9)

    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

Tests training, test, and accuracy of Logistic Regression model after preprocessing

In [7]:
def test_logreg():
    class TestRunModel(unittest.TestCase):
        def test_logreg_training(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = LogisticRegression(random_state=42, solver='sag')
            model.fit(X_train, y_train)
            self.assertIsNotNone(model)
            self.assertTrue(hasattr(model, 'intercept_'))
        def test_logreg_prediction(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = LogisticRegression(random_state=42, solver='sag')
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            self.assertEqual(len(predictions), len(X_test))
        def test_logreg_accuracy(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = LogisticRegression(random_state=42, solver='sag')
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)
            f1 = f1_score(y_test, predictions)
            self.assertGreaterEqual(accuracy, 0.9)
            self.assertGreaterEqual(f1, 0.9)

    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

Tests training, test, and accuracy of Random Forest model after preprocessing

In [8]:
def test_rfc():
    class TestRunModel(unittest.TestCase):
        def test_rfc_training(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = RandomForestClassifier(random_state=42)
            model.fit(X_train, y_train)
            self.assertIsNotNone(model)
            self.assertTrue(hasattr(model, 'max_depth'))
        def test_rfc_prediction(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = RandomForestClassifier(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            self.assertEqual(len(predictions), len(X_test))
        def test_rfc_accuracy(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = RandomForestClassifier(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)
            f1 = f1_score(y_test, predictions)
            self.assertGreaterEqual(accuracy, 0.9)
            self.assertGreaterEqual(f1, 0.9)

    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

Tests training, test, and accuracy of Decision Trees model after preprocessing

In [9]:
def test_dtc():
    class TestRunModel(unittest.TestCase):
        def test_dtc_training(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = DecisionTreeClassifier(random_state=42)
            model.fit(X_train, y_train)
            self.assertIsNotNone(model)
            self.assertTrue(hasattr(model, 'max_depth'))
        def test_dtc_prediction(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = DecisionTreeClassifier(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            self.assertEqual(len(predictions), len(X_test))
        def test_dtc_accuracy(self):
            file_path = 'https://raw.githubusercontent.com/tsimhadri-ews/internproject/malware-detection-0/src/MalwareData.csv'
            df = pd.read_csv(file_path, sep='|')
            df = malwarefunctions.preprocess(df)
            X_train, X_test, y_train, y_test = malwarefunctions.traintest_split(df)
            model = DecisionTreeClassifier(random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)
            f1 = f1_score(y_test, predictions)
            self.assertGreaterEqual(accuracy, 0.9)
            self.assertGreaterEqual(f1, 0.9)

    suite = unittest.TestLoader().loadTestsFromTestCase(TestRunModel)
    unittest.TextTestRunner(verbosity=4,stream=sys.stderr).run(suite)

Testing each individual component to make sure each component passes all of its tests

In [10]:
test_preprocess()
test_gbc()
test_nb()
test_ann()
test_svm()
test_logreg()
test_rfc()
test_dtc()

test_components_drop (__main__.test_preprocess.<locals>.TestRunModel) ... ok
test_df_shape (__main__.test_preprocess.<locals>.TestRunModel) ... ok
test_normalization (__main__.test_preprocess.<locals>.TestRunModel) ... ok
test_value_counts (__main__.test_preprocess.<locals>.TestRunModel) ... ok

----------------------------------------------------------------------
Ran 4 tests in 8.067s

OK
test_gbc_accuracy (__main__.test_gbc.<locals>.TestRunModel) ... ok
test_gbc_prediction (__main__.test_gbc.<locals>.TestRunModel) ... ok
test_gbc_training (__main__.test_gbc.<locals>.TestRunModel) ... ok

----------------------------------------------------------------------
Ran 3 tests in 121.372s

OK
test_nb_accuracy (__main__.test_nb.<locals>.TestRunModel) ... ok
test_nb_prediction (__main__.test_nb.<locals>.TestRunModel) ... ok
test_nb_training (__main__.test_nb.<locals>.TestRunModel) ... ok

----------------------------------------------------------------------
Ran 3 tests in 8.212s

OK
test_ann

863/863 [==============================] - 1s 945us/step


ok
test_ann_training (__main__.test_ann.<locals>.TestRunModel) ... ok

----------------------------------------------------------------------
Ran 3 tests in 114.841s

OK
test_svm_accuracy (__main__.test_svm.<locals>.TestRunModel) ... ok
test_svm_prediction (__main__.test_svm.<locals>.TestRunModel) ... ok
test_svm_training (__main__.test_svm.<locals>.TestRunModel) ... ok

----------------------------------------------------------------------
Ran 3 tests in 202.664s

OK
test_logreg_accuracy (__main__.test_logreg.<locals>.TestRunModel) ... /usr/local/lib/python3.8/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
ok
test_logreg_prediction (__main__.test_logreg.<locals>.TestRunModel) ... /usr/local/lib/python3.8/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
ok
test_logreg_training (__main__.

Containerizes each component and saves them as .yaml files to later be used in a kubeflow pipeline

In [11]:
from kfp import components

test_preprocess_op = components.func_to_container_op(func=test_preprocess, output_component_file='test_preprocess.yaml', base_image='python:3.7', packages_to_install=['pandas','scikit-learn', 'kfp', 'numpy'])
test_gbc_op = components.func_to_container_op(func=test_gbc, output_component_file='test_gbc.yaml', base_image='python:3.7', packages_to_install=['pandas','scikit-learn', 'kfp', 'numpy'])
test_nb_op = components.func_to_container_op(func=test_nb, output_component_file='test_nb.yaml', base_image='python:3.7', packages_to_install=['pandas','scikit-learn', 'kfp', 'numpy'])
test_ann_op = components.func_to_container_op(func=test_ann, output_component_file='test_ann.yaml', base_image='python:3.7', packages_to_install=['pandas','scikit-learn', 'kfp', 'numpy'])
test_svm_op = components.func_to_container_op(func=test_svm, output_component_file='test_svm.yaml', base_image='python:3.7', packages_to_install=['pandas','scikit-learn', 'kfp', 'numpy'])
test_logreg_op = components.func_to_container_op(func=test_logreg, output_component_file='test_logreg.yaml', base_image='python:3.7', packages_to_install=['pandas','scikit-learn', 'kfp', 'numpy'])
test_rfc_op = components.func_to_container_op(func=test_rfc, output_component_file='test_rfc.yaml', base_image='python:3.7', packages_to_install=['pandas','scikit-learn', 'kfp', 'numpy'])
test_dtc_op = components.func_to_container_op(func=test_dtc, output_component_file='test_dtc.yaml', base_image='python:3.7', packages_to_install=['pandas','scikit-learn', 'kfp', 'numpy'])

In [ ]:
from github import Auth
from github import Github
import nbformat

auth = Auth.Token("access_token")
g = Github(auth=auth)
g.get_user().login


repo = g.get_repo("tsimhadri-ews/internproject")
file_content = repo.get_contents("blob/malware-detection-0/src/malwarefunctions.ipynb")